In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

data_dir = Path("../model_outputs")

# Load historical+SSP245 (r1) and landmask
ds  = xr.open_dataset(data_dir / "tas_Amon_MPI-ESM1-2-LR_historical-ssp245_r1i1p1f1_g025_185001-210012.nc")
lm  = xr.open_dataset(data_dir / "landmask.nc")

tas = ds["tas"]

# sftlf values are 0.0 (ocean) and 1.0 (land) despite metadata saying 0–100%
land_mask  = lm["sftlf"] > 0.5
ocean_mask = lm["sftlf"] <= 0.5

# Area weights: cos(lat) accounts for smaller grid cells near poles
lat_weights = np.cos(np.deg2rad(ds.lat))
weights_3d  = lat_weights.broadcast_like(tas)

def masked_annual_mean(ta, mask):
    w = weights_3d.where(mask)
    return ((ta.where(mask) * w).sum(dim=["lat", "lon"]) / w.sum(dim=["lat", "lon"]) - 273.15).resample(time="YE").mean()

global_mean = (tas.weighted(lat_weights).mean(dim=["lat", "lon"]) - 273.15).resample(time="YE").mean()
land_mean   = masked_annual_mean(tas, land_mask)
ocean_mean  = masked_annual_mean(tas, ocean_mask)

years = global_mean.time.dt.year.values

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(years, global_mean.values, color="black",      lw=1.5, label="Global mean")
ax.plot(years, ocean_mean.values,  color="steelblue",  lw=1.5, label="Ocean surface")
ax.plot(years, land_mean.values,   color="seagreen",   lw=1.5, label="Land surface")

ax.axvline(2014, color="gray", lw=1, linestyle="--", label="historical → SSP2-4.5")

ax.set_xlabel("Year")
ax.set_ylabel("Temperature (°C)")
ax.set_title("MPI-ESM1-2-LR  |  Annual Mean Near-Surface Temperature  |  Historical + SSP2-4.5 (r1)")
ax.legend()
ax.grid(alpha=0.3)

fig.text(0.99, 0.01, "Nasrul Huda", ha="right", va="bottom", fontsize=9, color="gray")

plt.tight_layout()
plt.savefig("global_land_ocean_tas.pdf", format="pdf")
plt.show()